Long Training

In [7]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
import heapq
import itertools
import time
from enum import IntEnum
import os

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback


class Actions(IntEnum):
    HOLD = 0
    BUY = 1
    SELL = 2

OBSERVATION_DIMS = 14
INITIAL_CASH = 100_000.0
MAX_STEPS_PER_EPISODE = 2000 

# --- CRITICAL UPDATES ---
TRANSACTION_COST = 0.0 
INVENTORY_RISK_COEFF = 0.0  # Removed fear of holding
DOWNSIDE_PENALTY_MULT = 0.0 # Removed fear of losing money
TRADE_BONUS = 1.0           # Bribe the agent to act!


class AdvancedOrderBook:
    def __init__(self):
        self.bids = []  
        self.asks = []  
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty


class TradingEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self):
        super(TradingEnv, self).__init__()
        self.action_space = spaces.Discrete(len(Actions))
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, 
            shape=(OBSERVATION_DIMS,), dtype=np.float32
        )
        self.engine = None
        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.max_portfolio_value = INITIAL_CASH
        self.current_step = 0
        self.mid_price_history = [] 

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.engine = AdvancedOrderBook()
        start_price = 100.0
        for i in range(1, 6):
            self.engine.submit_order('buy', 10, start_price - i*0.5, 'limit')
            self.engine.submit_order('sell', 10, start_price + i*0.5, 'limit')
        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.max_portfolio_value = INITIAL_CASH
        self.current_step = 0
        self.mid_price_history = [start_price] * 10 
        return self._get_observation(), {}

    def step(self, action):
        prev_val = self.portfolio_value
        
        # Action
        if action == Actions.BUY and self.cash > 0:
            rem = self.engine.submit_order('buy', 1, order_type='market')
            if rem == 0: 
                fill_price = self.engine.trades[-1]['price']
                self.inventory += 1
                self.cash -= fill_price
        elif action == Actions.SELL:
            rem = self.engine.submit_order('sell', 1, order_type='market')
            if rem == 0: 
                fill_price = self.engine.trades[-1]['price']
                self.inventory -= 1
                self.cash += abs(fill_price)

        # Market Move (MASSIVE VOLATILITY)
        mid = self._get_mid_price()
        shock = np.random.normal(0, 5.0) # Huge swings to make profit obvious
        self.engine.submit_order('buy', 10, round(mid + shock - 0.5, 2), 'limit')
        self.engine.submit_order('sell', 10, round(mid + shock + 0.5, 2), 'limit')

        # Update PnL
        mid = self._get_mid_price()
        self.portfolio_value = self.cash + (self.inventory * mid)
        if self.portfolio_value > self.max_portfolio_value:
            self.max_portfolio_value = self.portfolio_value
            
        # Reward
        delta_pnl = self.portfolio_value - prev_val
        
        # Bribe logic:
        participation_reward = 0
        if action != Actions.HOLD:
            participation_reward = TRADE_BONUS # +1.0 for doing anything
        
        reward = delta_pnl + participation_reward
        
        self.current_step += 1
        terminated = self.portfolio_value <= 0 
        truncated = self.current_step >= MAX_STEPS_PER_EPISODE
        
        return self._get_observation(), reward, terminated, truncated, {'pnl': self.portfolio_value - INITIAL_CASH}

    def _get_mid_price(self):
        best_bid = -self.engine.bids[0][0] if self.engine.bids else 100.0
        best_ask = self.engine.asks[0][0] if self.engine.asks else 100.0
        mid = (best_bid + best_ask) / 2
        self.mid_price_history.append(mid)
        if len(self.mid_price_history) > 20: self.mid_price_history.pop(0)
        return mid

    def _get_observation(self):
        mid = self._get_mid_price()
        bids = [-x[0] for x in heapq.nsmallest(5, self.engine.bids)] if self.engine.bids else []
        asks = [x[0] for x in heapq.nsmallest(5, self.engine.asks)] if self.engine.asks else []
        while len(bids) < 5: bids.append(mid)
        while len(asks) < 5: asks.append(mid)
        
        norm_bids = [(p - mid)/mid for p in bids]
        norm_asks = [(p - mid)/mid for p in asks]
        norm_inv = self.inventory / 100.0
        norm_cash = self.cash / 1_000_000.0
        spread = (asks[0] - bids[0]) / mid
        vol = np.std(self.mid_price_history) / mid if len(self.mid_price_history) > 1 else 0
        
        return np.array(norm_bids + norm_asks + [norm_inv, norm_cash, spread, vol], dtype=np.float32)

# 4. LOGGING CALLBACK

class SanityCheckCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(SanityCheckCallback, self).__init__(verbose)
        self.action_counts = {0: 0, 1: 0, 2: 0} 
        
    def _on_step(self) -> bool:
        actions = self.locals.get("actions")
        if actions is not None:
            self.action_counts[int(actions[0])] += 1
        return True
    
    def _on_rollout_end(self) -> None:
        total = sum(self.action_counts.values())
        if total > 0:
            print(f"   >>> Activity: HOLD={self.action_counts[0]/total:.1%} | BUY={self.action_counts[1]/total:.1%} | SELL={self.action_counts[2]/total:.1%}")
        self.action_counts = {0: 0, 1: 0, 2: 0}

# 5. RUNNER

def run_sanity_check():
    print("--- Week 3 Day 5: 'Hyper-Active' Sanity Check ---")
    
    log_dir = "logs/"
    os.makedirs(log_dir, exist_ok=True)
    
    env = TradingEnv()
    env = Monitor(env, log_dir) 
    
    model = PPO("MlpPolicy", env, verbose=0, learning_rate=0.0003, ent_coef=0.05)
    
    print("Starting Training (With Trade Bribe +1.0)...")
    callback = SanityCheckCallback()
    model.learn(total_timesteps=50000, callback=callback)
    
    print("\nTraining Complete.")
    
    obs, _ = env.reset()
    done = False
    total_reward = 0
    actions_taken = []
    
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        actions_taken.append(action)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated
        
    buy_count = actions_taken.count(Actions.BUY)
    sell_count = actions_taken.count(Actions.SELL)
    
    print(f"\n--- VERDICT ---")
    print(f"Total Reward: {total_reward:.2f}")
    print(f"Trades Executed: {buy_count + sell_count}")
    
    if buy_count + sell_count == 0:
        print("FAIL: Even with a bribe, it won't trade. Something is fundamentally broken.")
    else:
        print("PASS: Agent is trading wildly! (We will tame it later).")

if __name__ == "__main__":
    run_sanity_check()

--- Week 3 Day 5: 'Hyper-Active' Sanity Check ---
Starting Training (With Trade Bribe +1.0)...
   >>> Activity: HOLD=32.8% | BUY=32.7% | SELL=34.6%
   >>> Activity: HOLD=33.7% | BUY=36.7% | SELL=29.5%
   >>> Activity: HOLD=35.6% | BUY=31.4% | SELL=32.9%
   >>> Activity: HOLD=28.7% | BUY=35.4% | SELL=35.9%
   >>> Activity: HOLD=14.4% | BUY=49.5% | SELL=36.1%
   >>> Activity: HOLD=14.0% | BUY=47.5% | SELL=38.6%
   >>> Activity: HOLD=24.4% | BUY=40.3% | SELL=35.3%
   >>> Activity: HOLD=8.4% | BUY=62.9% | SELL=28.7%
   >>> Activity: HOLD=8.7% | BUY=64.6% | SELL=26.7%
   >>> Activity: HOLD=10.4% | BUY=58.7% | SELL=30.8%
   >>> Activity: HOLD=10.8% | BUY=57.8% | SELL=31.3%
   >>> Activity: HOLD=10.9% | BUY=59.0% | SELL=30.0%
   >>> Activity: HOLD=11.0% | BUY=59.7% | SELL=29.2%
   >>> Activity: HOLD=9.1% | BUY=64.7% | SELL=26.2%
   >>> Activity: HOLD=8.6% | BUY=66.4% | SELL=25.0%
   >>> Activity: HOLD=7.3% | BUY=69.3% | SELL=23.4%
   >>> Activity: HOLD=8.5% | BUY=65.6% | SELL=25.9%
   >>> Act